In [0]:
import os,sys

In [0]:
sys.path.insert(0,os.path.abspath(os.path.join(os.getcwd(), '..')))

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
from utils.spark_utils import(
    add_silver_audit_columns,
    write_delta_overwrite,
    write_delta_append,
    dedup,
    drop_nulls,
    
)
from config.config import(
    BRONZE_CUSTOMERS,BRONZE_ORDERS,BRONZE_PRODUCTS,
    BRONZE_PATH,SILVER_PATH,GOLD_PATH,
)

## procesing customers

In [0]:
df_customers=spark.read.format("delta")\
        .option('inferSchema','true')\
        .load('/Volumes/salesdw/bronze/bronze_data/bronze_customers')   



In [0]:
df_customers.display()

In [0]:
df_customers=add_silver_audit_columns(df_customers)

In [0]:
df_customers=dedup(df_customers,['customer_id','email'],'registration_date')

In [0]:
df_customers=drop_nulls(df_customers,['customers_id','email','first_name'])

In [0]:
df_customers.display()

In [0]:
df_customers=df_customers.withColumn('customer_id',trim(upper(col("customer_id")))).withColumn('first_name',initcap(trim(col("first_name")))).withColumn('last_name',initcap(trim(col("last_name")))).withColumn('email',lower(trim(col("email")))).withColumn('phone',regexp_replace(col("phone"),r"[^0-9]",'')).withColumn('city',initcap(trim(col("city")))).withColumn('state',initcap(trim(col("state")))).withColumn('country',initcap(trim(col("country")))).withColumn('zip_code',trim(col('zip_code'))).withColumn('customer_segment',upper(trim(col("customer_segment")))) .withColumn("registration_date",  F.to_date(F.col("registration_date"), "yyyy-MM-dd")).withColumn("created_at",         F.current_timestamp()).withColumn("updated_at",         F.current_timestamp())
